In [ ]:
!pip install lapjv
!pip install igraph
!pip install leidenalg

import numpy as np
import scipy
from scipy import sparse
from lapjv import lapjv
import time
from itertools import combinations
import scipy.io
import pandas as pd
import csv
from google.colab import drive
from collections import defaultdict
from os import cpu_count
import re
from collections import defaultdict
import matplotlib.pyplot as plt
import igraph as ig
import leidenalg as la
import random
from typing import List



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for lapjv: filename=lapjv-1.3.27-cp312-cp312-linux_x86_64.whl size=142099 sha256=103e6d8a711b7f7f642563e2aba7624fc44d12da216aa9b1aafe24fa6a2bf64c
  Stored in directory: /root/.cache/pip/wheels/8a/b5/65/3cbb79ec56625bf795b818545064022bc30460016a31ab9b5f
Successfully built lapjv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 83.0 MB/s eta 0:00:00


In [ ]:
drive.mount('/content/drive')
%cd /content/drive/My\ Drive

Mounted at /content/drive
/content/drive/My Drive


In [ ]:
%cd /content/drive/MyDrive/
elegans = scipy.io.loadmat('elegansGraph.mat')
wiki_labels = scipy.io.loadmat('/content/drive/MyDrive/wiki_labels.mat')
wiki_adj = scipy.io.loadmat('wiki_adj.mat')

/content/drive/MyDrive


In [ ]:
graphA = wiki_adj['G_EN_Adj']
graphB = wiki_adj['G_FR_Adj']
graphA2 = graphA.copy()

wiki_labels = wiki_labels['wiki_labels'][0]
indices = wiki_labels.argsort()

graphA = wiki_adj['G_EN_Adj'][indices][:, indices]
graphB = wiki_adj['G_FR_Adj'][indices][:, indices]
comm = [370,270,191,212,220, 119]


In [ ]:
elegans_labels = scipy.io.loadmat('celegans_labels.mat')['cel_labels']
graphA = elegans['Achem'].toarray()
graphB = elegans['Agap'].toarray()
indices = elegans_labels.argsort()
graphA2 = graphA.copy()

graphA = graphA[indices][:, indices]
graphB = graphB[indices][:, indices]
comm = [118,83,78]

In [ ]:
!git clone https://github.com/Leron33/SeedGNN.git

fatal: destination path 'SeedGNN' already exists and is not an empty directory.


In [ ]:
import subprocess

requirements = """
python>=3.8
torch>=1.2.0
torch-geometric>=1.5.0
numpy>=1.20.1
scipy>=1.6.2
seaborn==0.11.2
graspologic>=1.0.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

subprocess.run(["pip", "install", "-r", "requirements.txt"])

CompletedProcess(args=['pip', 'install', '-r', 'requirements.txt'], returncode=1)

In [ ]:
pip install torch-geometric>=1.5.0

In [ ]:
pip install numpy>=1.20.1

In [ ]:
pip install scipy>=1.6.2

In [ ]:
pip install seaborn==0.11.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.8/292.8 kB 11.5 MB/s eta 0:00:00
  Attempting uninstall: seaborn
    Found existing installation: seaborn 0.13.2
    Uninstalling seaborn-0.13.2:
      Successfully uninstalled seaborn-0.13.2


In [ ]:
pip install graspologic>=1.0.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
cd SeedGNN/

/content/drive/MyDrive/SeedGNN


In [ ]:
!python TestER.py

Traceback (most recent call last):
  File "/content/drive/MyDrive/SeedGNN/TestER.py", line 7, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 427, in <module>
    from torch._C import *  # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 463, in _lock_unlock_module
KeyboardInterrupt
^C


In [ ]:
import sys
import os.path as osp
import numpy as np
import time

import argparse
import torch

from SeedGNN import SeedGNN
from MGCN import MGCN
from GMAlgorithms import MultiHop, PGM, SGM2

def SynGraph_from_numpy(A, B, theta, degreesort=False):
    """
    Parameters:
        A, B: numpy.ndarray, adjacency matrices of two graphs (n x n)
        theta: float, proportion of nodes to use as seeds (0 <= theta <= 1)
        degreesort: bool, if True, uses degree sort for permutation

    Returns:
        G1: torch.FloatTensor, permuted adjacency matrix for graph 1
        G2: torch.FloatTensor, adjacency matrix for graph 2 (possibly permuted if degreesort=True)
        seeds: list of two 1D torch.LongTensor, [indices in G1, matching indices in G2]
        truth: 1D torch.LongTensor, ground-truth mapping from rows of G1 to rows of G2
    """
    # Convert numpy arrays to torch tensors (float for adj, long for indices)
    A_tensor = torch.from_numpy(A).float()
    B_tensor = torch.from_numpy(B).float()
    n = A_tensor.size(0)
    assert B_tensor.size(0) == n and B_tensor.size(1) == n, "A and B must be n x n matrices"

    if degreesort:
        # Sort nodes by degree in A
        degree = A_tensor.sum(0)
        _, indices = torch.sort(degree, descending=True)
        G1 = A_tensor[indices][:, indices]
        G2 = B_tensor[indices][:, indices]
        truth = n - 1 - torch.arange(n)
    else:
        truth = torch.randperm(n)
        G1 = A_tensor[truth][:, truth]
        G2 = B_tensor

    numseeds = int(n * theta)
    indices = torch.randperm(n)[:numseeds]
    seeds = [indices, truth[indices]]

    return G1, G2, seeds, truth

def to_list_if_needed(value):
    # If value is already a list, return as is. Otherwise, try to convert using .tolist().
    if isinstance(value, list):
        return value
    # Check if value has .tolist() method (NumPy arrays, etc.)
    if hasattr(value, 'tolist'):
        return value.tolist()
    # Otherwise, return as is.
    return value



def generate_y(num_nodes, truth):
    oneton = torch.arange(num_nodes)
    return [oneton, truth]

def test(test_dataset):
    model.eval()

    total_correct = 0
    total_node = 0
    for data in test_dataset:


        G1 = data[0]
        G2 = data[1]
        seeds = data[2]
        truth = data[3]
        num_nodes = G1.shape[0]

        Y_L, _ = model(G1,G2,seeds)

        y = generate_y(num_nodes, truth)
        correct = model.acc(Y_L,y)
        total_correct += correct
        total_node += num_nodes
    return total_correct/total_node

def new_generate_array_comm(l, x):
    total = sum(l)
    ratio = x / total
    l2 = [min(int(size * ratio), size) for size in l]
    for i in range(len(l)):
        if l2[i] == l[i]:
            if l2[i] > 0:
                l2[i] -= 1
            else:
                l2[i] += 1
    current_sum = sum(l2)
    difference = x - current_sum

    fractional_parts = [(size * ratio) - int(size * ratio) for size in l]
    indices_sorted = sorted(
        range(len(l)),
        key=lambda i: (fractional_parts[i], -l[i]),
        reverse=True
    )

    for i in indices_sorted:
        if difference <= 0:
            break
        if abs(l2[i] + 1 - l[i]) >= 1 and l2[i] < l[i]:
            l2[i] += 1
            difference -= 1

    return l2


def run(graphA, graphB, L,Theta,Itera):
    n = len(graphA)
    seedgnn = torch.zeros(Itera,len(Theta))
    onehop6 = torch.zeros(Itera,len(Theta))
    twohop3 = torch.zeros(Itera,len(Theta))
    threehop2 = torch.zeros(Itera,len(Theta))
    pgm = torch.zeros(Itera,len(Theta))
    sgm = torch.zeros(Itera,len(Theta))
    mgcn = torch.zeros(len(Theta))

    for thetai, theta in enumerate(Theta):
        #print(theta)
        for itera in range(Itera):
            datasets = []
            G1, G2, seeds, truth = SynGraph_from_numpy(graphA,graphB,theta)
            datasets = [(G1, G2, seeds, truth)]
            eyes = torch.eye(n)
            G1 = G1
            G12 = (( ((torch.mm(G1, G1))>0).float() - G1 - eyes)>0).float()
            G22 = (( ((torch.mm(G2, G2))>0).float() - G2 - eyes)>0).float()
            G13 = (( ((torch.mm(G12, G1))>0).float() - G12 - G1 - eyes)>0).float()
            G23 = (( ((torch.mm(G22, G2))>0).float() - G22 - G2 - eyes)>0).float()

            SeedGNN
            seedgnn[itera,thetai] = test(datasets)

            # other algorithms
            result = seeds
            for _ in range(L):
                result = MultiHop(G1,G2,result)
            onehop6[itera,thetai] = sum((result[1]==truth).float())/n

            result = seeds
            for _ in range(int(L/2)):
                result = MultiHop(G12,G22,result)
            twohop3[itera,thetai] = sum((result[1]==truth).float())/n

            result = seeds
            for _ in range(int(L/3)):
                result = MultiHop(G13,G23,result)
            threehop2[itera,thetai] = sum((result[1]==truth).float())/n

            result = PGM(G1,G2,seeds)
            pgm[itera,thetai] = sum(result==truth)/n

            result = SGM2(G1,G2,seeds)
            sgm[itera,thetai] = sum((result==truth).float())/n


            result = MGCN(G1,G2,seeds)
            mgcn[thetai] = sum((result==truth).float())/n

    seedgnnstd, seedgnn = torch.std_mean(seedgnn,dim=0,unbiased=False)
    onehop6std,onehop6 = torch.std_mean(onehop6,dim=0,unbiased=False)
    twohop3std,twohop3 = torch.std_mean(twohop3,dim=0,unbiased=False)
    threehop2std,threehop2 = torch.std_mean(threehop2,dim=0,unbiased=False)
    pgmstd,pgm = torch.std_mean(pgm,dim=0,unbiased=False)
    sgmstd,sgm = torch.std_mean(sgm,dim=0,unbiased=False)


    theta = [round(i, 4) for i in to_list_if_needed(Theta)]
    seedgnn = [round(i, 4) for i in to_list_if_needed(seedgnn)]
    onehop6 = [round(i, 4) for i in to_list_if_needed(onehop6)]
    twohop3 = [round(i, 4) for i in to_list_if_needed(twohop3)]
    threehop2 = [round(i, 4) for i in to_list_if_needed(threehop2)]
    pgm = [round(i, 4) for i in to_list_if_needed(pgm)]
    sgm = [round(i, 4) for i in to_list_if_needed(sgm)]
    mgcn = [round(i, 4) for i in to_list_if_needed(mgcn)]

    torch.set_printoptions(precision=4)
    print('Accuracy')
    print('theta ='.ljust(10), theta)
    print('SeedGNN = '.ljust(10),seedgnn)
    print('onehop6 ='.ljust(10), onehop6)
    print('twohop3 ='.ljust(10), twohop3)
    print('threehop2 ='.ljust(10), threehop2)
    print('pgm = '.ljust(10), pgm)
    print('sgm = '.ljust(10), sgm)
    print('mgcn = '.ljust(10), mgcn)

    # seedgnnstd = [round(i,4) for i in (seedgnnstd).tolist()]
    # onehop6std = [round(i,4) for i in (onehop6std).tolist()]
    # twohop3std = [round(i,4) for i in (twohop3std).tolist()]
    # threehop2std = [round(i,4) for i in (threehop2std).tolist()]
    # pgmstd = [round(i,4) for i in (pgmstd).tolist()]
    # sgmstd = [round(i,4) for i in (sgmstd).tolist()]

    # print('seedgnnstd ='.ljust(13), seedgnnstd)
    # print('onehop6std ='.ljust(13), onehop6std)
    # print('twohop3std ='.ljust(13), twohop3std)
    # print('threehop2std ='.ljust(10), threehop2std)
    # print('pgmstd = '.ljust(13), pgmstd)
    # print('sgmstd = '.ljust(13), sgmstd)


if __name__ ==  '__main__':

    parser = argparse.ArgumentParser()
    parser.add_argument('--hid', type=int, default=4)
    parser.add_argument('--num_layers', type=int, default=6)

    args, unknown = parser.parse_known_args()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model = SeedGNN(num_layers=args.num_layers, hid=args.hid).to(device)

    path = "./model/SeedGNN-model.pth"
    model.load_state_dict(torch.load(path))
    for param in model.parameters():
        param.requires_grad = False

    start = time.time()
    L = 6
    number_of_seeds = [1, 5, 10, 20, 50, 75, 100, 150, 200]
    #Theta = [x / 279 for x in number_of_seeds]

    Itera = 10

    '''graphA = elegans['Achem'].toarray()
    graphB = elegans['Agap'].toarray()
    run(graphA, graphB, L,Theta,Itera)'''
    print('-----------------------------------------------')


    comm = [118,83,78]

    L = 8
    number_of_seeds = [1, 5, 10, 20, 50, 75, 100, 150, 200]
    Theta = [x / 279 for x in number_of_seeds]

    Itera = 10

    print("First community of Worm, 118 nodes")

    graphA = elegans['Achem'].toarray()[:118,:118]
    graphB = elegans['Agap'].toarray()[:118,:118]
    Theta = [x / 279 for x in number_of_seeds]

    run(graphA, graphB, L,Theta,Itera)

    print('-----------------------------------------------')

    print("Second community of Worm, 83 nodes")


    graphA = elegans['Achem'].toarray()[118:201,118:201]
    graphB = elegans['Agap'].toarray()[118:201,118:201]
    Theta = [x / 279 for x in number_of_seeds]

    run(graphA, graphB, L,Theta,Itera)

    print('-----------------------------------------------')

    print("Third community of Worm, 78 nodes")

    graphA = elegans['Achem'].toarray()[201:,201:]
    graphB = elegans['Agap'].toarray()[201:,201:]

    Theta = [x / 279 for x in number_of_seeds]

    run(graphA, graphB, L,Theta,Itera)


-----------------------------------------------
First community of Worm, 118 nodes
Accuracy
theta =    [0.0036, 0.0179, 0.0358, 0.0717, 0.1792, 0.2688, 0.3584, 0.5376, 0.7168]
SeedGNN =  [0.0085, 0.0322, 0.0432, 0.0805, 0.1907, 0.278, 0.3678, 0.5525, 0.7322]
onehop6 =  [0.0093, 0.0076, 0.0085, 0.0102, 0.0102, 0.0085, 0.0119, 0.0161, 0.022]
twohop3 =  [0.0042, 0.011, 0.0051, 0.0102, 0.0059, 0.0059, 0.0034, 0.0042, 0.0093]
threehop2 = [0.0042, 0.0034, 0.0042, 0.0059, 0.0068, 0.0076, 0.0068, 0.0085, 0.0085]
pgm =      [0.0085, 0.0093, 0.0076, 0.0119, 0.0144, 0.0136, 0.0169, 0.0178, 0.0153]
sgm =      [0.0085, 0.0288, 0.039, 0.0763, 0.189, 0.2763, 0.3678, 0.5534, 0.7271]
mgcn =     [0.0, 0.0254, 0.0424, 0.0678, 0.178, 0.2627, 0.3559, 0.5339, 0.7203]
-----------------------------------------------
Second community of Worm, 83 nodes
Accuracy
theta =    [0.0036, 0.0179, 0.0358, 0.0717, 0.1792, 0.2688, 0.3584, 0.5376, 0.7168]
SeedGNN =  [0.012, 0.0241, 0.0386, 0.0723, 0.1843, 0.2964, 0.3699, 0

In [ ]:
graphA = wiki_adj['G_EN_Adj']
graphB = wiki_adj['G_FR_Adj']
graphA2 = graphA.copy()

wiki_labels = wiki_labels['wiki_labels'][0]
indices = wiki_labels.argsort()

graphA = wiki_adj['G_EN_Adj'][indices][:, indices]
graphB = wiki_adj['G_FR_Adj'][indices][:, indices]
comm = [370,270,191,212,220, 119]

In [ ]:
#elegans
n_seeds = [1, 5, 10, 20, 50, 75, 100, 150, 200]
CESSNA_true_community = [0.0235, 0.0376, 0.0455, 0.0903, 0.2038, 0.2981, 0.3905, 0.5654, 0.7367]
CESSNA_Leiden_community = [0.1032, 0.1259, 0.1515, 0.1831, 0.2695, 0.3639, 0.4675, 0.6209, 0.7763]
SeedGNN =  [0.0072, 0.023, 0.0415, 0.0842, 0.1894, 0.2811, 0.3772, 0.5594, 0.7348]
onehop6 =  [0.0045, 0.0051, 0.0033, 0.006, 0.0057, 0.0105, 0.0108, 0.0105, 0.0122]
twohop3 =  [0.0027, 0.0027, 0.0036, 0.0027, 0.003, 0.0033, 0.0015, 0.0048, 0.003]
threehop2 = [0.0024, 0.0027, 0.0018, 0.0027, 0.0033, 0.0039, 0.0039, 0.0024, 0.0024]
pgm =      [0.0048, 0.0051, 0.0036, 0.0066, 0.0078, 0.0078, 0.0131, 0.0191, 0.0161]
sgm =      [0.0075, 0.0212, 0.04, 0.0797, 0.1932, 0.2879, 0.3754, 0.5606, 0.7372]
mgcn =     [0.0108, 0.0251, 0.0394, 0.0717, 0.1828, 0.276, 0.362, 0.5376, 0.7168]

In [ ]:
n_seeds = [1, 5, 10, 20, 50, 75, 100, 150, 200]
CESSNA_raw1 = [0.02,0.02,0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.11]
CESSNA_raw2 = [0.10,0.11,0.12,0.12,0.11,0.13,0.17,0.18,0.21,0.24]
CESSNA_true_community = [round(float(a*(279-b)+b)/279,4) for a, b in zip(CESSNA_raw1, n_seeds)]
CESSNA_Leiden_community = [round(float(a*(279-b)+b)/279,4) for a, b in zip(CESSNA_raw2, n_seeds)]

In [ ]:
CESSNA_Leiden_community

[0.1032, 0.1259, 0.1515, 0.1831, 0.2695, 0.3639, 0.4675, 0.6209, 0.7763]

In [ ]:
import os
import numpy as np

base_path = '/content/drive/MyDrive/'
rhos = [round(1.0 - 0.1*i, 1) for i in range(11)]  # [1.0, 0.9, ..., 0.0]
idxs = range(9, -1, -1)  # 9 down to 0

'''matrices = {}

for rho in rhos:
    for idx in idxs:
        #print(idx)
        for label in ['A', 'B']:
            filename = f'dense_graphs/P_matrix_rho_{rho}_idx_{idx}_{label}.npy'
            file_path = os.path.join(base_path, filename)
            if os.path.isfile(file_path):
                matrices[(rho, idx, label)] = np.load(file_path)'''

matrices3 = {}
for rho in rhos:
    for idx in idxs:
        for label in ['A', 'B']:
            filename = f'less_dense_graphs2/P3_matrix_rho_{rho}_idx_{idx}_{label}.npy'
            file_path = os.path.join(base_path, filename)
            if os.path.isfile(file_path):
                matrices3[(rho, idx, label)] = np.load(file_path)

In [ ]:
import sys
import os.path as osp
import numpy as np
import time

import argparse
import torch

from SeedGNN import SeedGNN
from MGCN import MGCN
from GMAlgorithms import MultiHop, PGM, SGM2

def SynGraph_from_numpy(A, B, theta, degreesort=False):
    """
    Parameters:
        A, B: numpy.ndarray, adjacency matrices of two graphs (n x n)
        theta: float, proportion of nodes to use as seeds (0 <= theta <= 1)
        degreesort: bool, if True, uses degree sort for permutation

    Returns:
        G1: torch.FloatTensor, permuted adjacency matrix for graph 1
        G2: torch.FloatTensor, adjacency matrix for graph 2 (possibly permuted if degreesort=True)
        seeds: list of two 1D torch.LongTensor, [indices in G1, matching indices in G2]
        truth: 1D torch.LongTensor, ground-truth mapping from rows of G1 to rows of G2
    """
    # Convert numpy arrays to torch tensors (float for adj, long for indices)
    A_tensor = torch.from_numpy(A).float()
    B_tensor = torch.from_numpy(B).float()
    n = A_tensor.size(0)
    assert B_tensor.size(0) == n and B_tensor.size(1) == n, "A and B must be n x n matrices"

    if degreesort:
        # Sort nodes by degree in A
        degree = A_tensor.sum(0)
        _, indices = torch.sort(degree, descending=True)
        G1 = A_tensor[indices][:, indices]
        G2 = B_tensor[indices][:, indices]
        truth = n - 1 - torch.arange(n)
    else:
        truth = torch.randperm(n)
        G1 = A_tensor[truth][:, truth]
        G2 = B_tensor

    numseeds = int(n * theta)
    indices = torch.randperm(n)[:numseeds]
    seeds = [indices, truth[indices]]

    return G1, G2, seeds, truth

def to_list_if_needed(value):
    # If value is already a list, return as is. Otherwise, try to convert using .tolist().
    if isinstance(value, list):
        return value
    # Check if value has .tolist() method (NumPy arrays, etc.)
    if hasattr(value, 'tolist'):
        return value.tolist()
    # Otherwise, return as is.
    return value



def generate_y(num_nodes, truth):
    oneton = torch.arange(num_nodes)
    return [oneton, truth]

def test(test_dataset):
    model.eval()

    total_correct = 0
    total_node = 0
    for data in test_dataset:


        G1 = data[0]
        G2 = data[1]
        seeds = data[2]
        truth = data[3]
        num_nodes = G1.shape[0]

        Y_L, _ = model(G1,G2,seeds)

        y = generate_y(num_nodes, truth)
        correct = model.acc(Y_L,y)
        total_correct += correct
        total_node += num_nodes
    return total_correct/total_node

def new_generate_array_comm(l, x):
    total = sum(l)
    ratio = x / total
    l2 = [min(int(size * ratio), size) for size in l]
    for i in range(len(l)):
        if l2[i] == l[i]:
            if l2[i] > 0:
                l2[i] -= 1
            else:
                l2[i] += 1
    current_sum = sum(l2)
    difference = x - current_sum

    fractional_parts = [(size * ratio) - int(size * ratio) for size in l]
    indices_sorted = sorted(
        range(len(l)),
        key=lambda i: (fractional_parts[i], -l[i]),
        reverse=True
    )

    for i in indices_sorted:
        if difference <= 0:
            break
        if abs(l2[i] + 1 - l[i]) >= 1 and l2[i] < l[i]:
            l2[i] += 1
            difference -= 1

    return l2


def run(graphA, graphB, L,Theta,Itera):
    n = len(graphA)
    seedgnn = torch.zeros(Itera,len(Theta))
    onehop6 = torch.zeros(Itera,len(Theta))
    twohop3 = torch.zeros(Itera,len(Theta))
    threehop2 = torch.zeros(Itera,len(Theta))
    pgm = torch.zeros(Itera,len(Theta))
    sgm = torch.zeros(Itera,len(Theta))
    mgcn = torch.zeros(len(Theta))

    for thetai, theta in enumerate(Theta):
        for itera in range(Itera):
            datasets = []
            G1, G2, seeds, truth = SynGraph_from_numpy(graphA,graphB,theta)
            datasets = [(G1, G2, seeds, truth)]
            eyes = torch.eye(n)
            G1 = G1
            G12 = (( ((torch.mm(G1, G1))>0).float() - G1 - eyes)>0).float()
            G22 = (( ((torch.mm(G2, G2))>0).float() - G2 - eyes)>0).float()
            G13 = (( ((torch.mm(G12, G1))>0).float() - G12 - G1 - eyes)>0).float()
            G23 = (( ((torch.mm(G22, G2))>0).float() - G22 - G2 - eyes)>0).float()

            SeedGNN
            seedgnn[itera,thetai] = test(datasets)

            # other algorithms
            result = seeds
            for _ in range(L):
                result = MultiHop(G1,G2,result)
            onehop6[itera,thetai] = sum((result[1]==truth).float())/n

            result = seeds
            for _ in range(int(L/2)):
                result = MultiHop(G12,G22,result)
            twohop3[itera,thetai] = sum((result[1]==truth).float())/n

            result = seeds
            for _ in range(int(L/3)):
                result = MultiHop(G13,G23,result)
            threehop2[itera,thetai] = sum((result[1]==truth).float())/n

            result = PGM(G1,G2,seeds)
            pgm[itera,thetai] = sum(result==truth)/n

            result = SGM2(G1,G2,seeds)
            sgm[itera,thetai] = sum((result==truth).float())/n


            result = MGCN(G1,G2,seeds)
            mgcn[thetai] = sum((result==truth).float())/n

    seedgnnstd, seedgnn = torch.std_mean(seedgnn,dim=0,unbiased=False)
    onehop6std,onehop6 = torch.std_mean(onehop6,dim=0,unbiased=False)
    twohop3std,twohop3 = torch.std_mean(twohop3,dim=0,unbiased=False)
    threehop2std,threehop2 = torch.std_mean(threehop2,dim=0,unbiased=False)
    pgmstd,pgm = torch.std_mean(pgm,dim=0,unbiased=False)
    sgmstd,sgm = torch.std_mean(sgm,dim=0,unbiased=False)


    theta = [round(i, 4) for i in to_list_if_needed(Theta)]
    seedgnn = [round(i, 4) for i in to_list_if_needed(seedgnn)]
    onehop6 = [round(i, 4) for i in to_list_if_needed(onehop6)]
    twohop3 = [round(i, 4) for i in to_list_if_needed(twohop3)]
    threehop2 = [round(i, 4) for i in to_list_if_needed(threehop2)]
    pgm = [round(i, 4) for i in to_list_if_needed(pgm)]
    sgm = [round(i, 4) for i in to_list_if_needed(sgm)]
    mgcn = [round(i, 4) for i in to_list_if_needed(mgcn)]

    torch.set_printoptions(precision=4)
    print('Accuracy')
    print('theta ='.ljust(10), theta)
    print('SeedGNN = '.ljust(10),seedgnn)
    print('onehop6 ='.ljust(10), onehop6)
    print('twohop3 ='.ljust(10), twohop3)
    print('threehop2 ='.ljust(10), threehop2)
    print('pgm = '.ljust(10), pgm)
    print('sgm = '.ljust(10), sgm)
    print('mgcn = '.ljust(10), mgcn)


if __name__ ==  '__main__':

    parser = argparse.ArgumentParser()
    parser.add_argument('--hid', type=int, default=4)
    parser.add_argument('--num_layers', type=int, default=6)

    args, unknown = parser.parse_known_args()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model = SeedGNN(num_layers=args.num_layers, hid=args.hid).to(device)

    path = "./model/SeedGNN-model.pth"
    model.load_state_dict(torch.load(path))
    for param in model.parameters():
        param.requires_grad = False

    start = time.time()
    L = 6

    Theta = [x / 300 for x in number_of_seeds]

    Itera = 10

    graphA = matrices3[(0.6, 0, 'A')]
    graphB = matrices3[(0.6, 0, 'B')]

    number_of_seeds = [1, 5, 10, 20, 50, 75, 100, 150, 200]
    comm = [100,100,100]
    cum_comm = [0]

    for size in comm:
        cum_comm.append(cum_comm[-1]+size)


    for k in range(len(comm)):
        c_start = cum_comm[k]
        c_end = cum_comm[k+1]
        size = comm[k]
        print(f"Community {k+1} size: " + str(graphA[c_start:c_end, c_start:c_end].shape))
        Theta = [x / 300 for x in number_of_seeds]
        graphA_new = graphA[c_start:c_end, c_start:c_end]
        graphB_new = graphB[c_start:c_end, c_start:c_end]
        run(graphA_new, graphB_new, L, Theta, Itera)
        print('-----------------------------------------------')

Community 1 size: (100, 100)
Accuracy
theta =    [0.0033, 0.0167, 0.0333, 0.0667, 0.1667, 0.25, 0.3333, 0.5, 0.6667]
SeedGNN =  [0.012, 0.025, 0.062, 0.219, 1.0, 1.0, 1.0, 1.0, 1.0]
onehop6 =  [0.021, 0.032, 0.057, 0.171, 1.0, 1.0, 1.0, 1.0, 1.0]
twohop3 =  [0.021, 0.028, 0.053, 0.115, 0.998, 1.0, 1.0, 1.0, 1.0]
threehop2 = [0.009, 0.01, 0.01, 0.015, 0.007, 0.015, 0.007, 0.004, 0.01]
pgm =      [0.01, 0.016, 0.016, 0.03, 0.051, 0.083, 0.123, 0.234, 0.401]
sgm =      [0.051, 0.08, 0.248, 0.986, 1.0, 1.0, 1.0, 1.0, 1.0]
mgcn =     [0.01, 0.02, 0.03, 0.06, 0.17, 0.26, 0.33, 0.5, 0.66]
-----------------------------------------------
Community 2 size: (100, 100)
Accuracy
theta =    [0.0033, 0.0167, 0.0333, 0.0667, 0.1667, 0.25, 0.3333, 0.5, 0.6667]
SeedGNN =  [0.012, 0.026, 0.066, 0.273, 1.0, 1.0, 1.0, 1.0, 1.0]
onehop6 =  [0.019, 0.025, 0.051, 0.14, 1.0, 1.0, 1.0, 1.0, 1.0]
twohop3 =  [0.02, 0.027, 0.03, 0.084, 1.0, 1.0, 1.0, 1.0, 1.0]
threehop2 = [0.009, 0.017, 0.01, 0.008, 0.01, 0.011, 0

In [ ]:
import sys
import os.path as osp
import numpy as np
import time

import argparse
import torch

from SeedGNN import SeedGNN
from MGCN import MGCN
from GMAlgorithms import MultiHop, PGM, SGM2

def SynGraph_from_numpy(A, B, theta, degreesort=False):
    """
    Parameters:
        A, B: numpy.ndarray, adjacency matrices of two graphs (n x n)
        theta: float, proportion of nodes to use as seeds (0 <= theta <= 1)
        degreesort: bool, if True, uses degree sort for permutation

    Returns:
        G1: torch.FloatTensor, permuted adjacency matrix for graph 1
        G2: torch.FloatTensor, adjacency matrix for graph 2 (possibly permuted if degreesort=True)
        seeds: list of two 1D torch.LongTensor, [indices in G1, matching indices in G2]
        truth: 1D torch.LongTensor, ground-truth mapping from rows of G1 to rows of G2
    """
    # Convert numpy arrays to torch tensors (float for adj, long for indices)
    A_tensor = torch.from_numpy(A).float()
    B_tensor = torch.from_numpy(B).float()
    n = A_tensor.size(0)
    assert B_tensor.size(0) == n and B_tensor.size(1) == n, "A and B must be n x n matrices"

    if degreesort:
        # Sort nodes by degree in A
        degree = A_tensor.sum(0)
        _, indices = torch.sort(degree, descending=True)
        G1 = A_tensor[indices][:, indices]
        G2 = B_tensor[indices][:, indices]
        truth = n - 1 - torch.arange(n)
    else:
        truth = torch.randperm(n)
        G1 = A_tensor[truth][:, truth]
        G2 = B_tensor

    numseeds = int(n * theta)
    indices = torch.randperm(n)[:numseeds]
    seeds = [indices, truth[indices]]

    return G1, G2, seeds, truth

def to_list_if_needed(value):
    # If value is already a list, return as is. Otherwise, try to convert using .tolist().
    if isinstance(value, list):
        return value
    # Check if value has .tolist() method (NumPy arrays, etc.)
    if hasattr(value, 'tolist'):
        return value.tolist()
    # Otherwise, return as is.
    return value



def generate_y(num_nodes, truth):
    oneton = torch.arange(num_nodes)
    return [oneton, truth]

def test(test_dataset):
    model.eval()

    total_correct = 0
    total_node = 0
    for data in test_dataset:


        G1 = data[0]
        G2 = data[1]
        seeds = data[2]
        truth = data[3]
        num_nodes = G1.shape[0]

        Y_L, _ = model(G1,G2,seeds)

        y = generate_y(num_nodes, truth)
        correct = model.acc(Y_L,y)
        total_correct += correct
        total_node += num_nodes
    return total_correct/total_node

def new_generate_array_comm(l, x):
    total = sum(l)
    ratio = x / total
    l2 = [min(int(size * ratio), size) for size in l]
    for i in range(len(l)):
        if l2[i] == l[i]:
            if l2[i] > 0:
                l2[i] -= 1
            else:
                l2[i] += 1
    current_sum = sum(l2)
    difference = x - current_sum

    fractional_parts = [(size * ratio) - int(size * ratio) for size in l]
    indices_sorted = sorted(
        range(len(l)),
        key=lambda i: (fractional_parts[i], -l[i]),
        reverse=True
    )

    for i in indices_sorted:
        if difference <= 0:
            break
        if abs(l2[i] + 1 - l[i]) >= 1 and l2[i] < l[i]:
            l2[i] += 1
            difference -= 1

    return l2


def run(graphA, graphB, L,Theta,Itera):
    n = len(graphA)
    seedgnn = torch.zeros(Itera,len(Theta))
    onehop6 = torch.zeros(Itera,len(Theta))
    twohop3 = torch.zeros(Itera,len(Theta))
    threehop2 = torch.zeros(Itera,len(Theta))
    pgm = torch.zeros(Itera,len(Theta))
    sgm = torch.zeros(Itera,len(Theta))
    mgcn = torch.zeros(len(Theta))

    for thetai, theta in enumerate(Theta):
        for itera in range(Itera):
            datasets = []
            G1, G2, seeds, truth = SynGraph_from_numpy(graphA,graphB,theta)
            datasets = [(G1, G2, seeds, truth)]
            eyes = torch.eye(n)
            G1 = G1
            G12 = (( ((torch.mm(G1, G1))>0).float() - G1 - eyes)>0).float()
            G22 = (( ((torch.mm(G2, G2))>0).float() - G2 - eyes)>0).float()
            G13 = (( ((torch.mm(G12, G1))>0).float() - G12 - G1 - eyes)>0).float()
            G23 = (( ((torch.mm(G22, G2))>0).float() - G22 - G2 - eyes)>0).float()

            SeedGNN
            seedgnn[itera,thetai] = test(datasets)

            # other algorithms
            result = seeds
            for _ in range(L):
                result = MultiHop(G1,G2,result)
            onehop6[itera,thetai] = sum((result[1]==truth).float())/n

            result = seeds
            for _ in range(int(L/2)):
                result = MultiHop(G12,G22,result)
            twohop3[itera,thetai] = sum((result[1]==truth).float())/n

            result = seeds
            for _ in range(int(L/3)):
                result = MultiHop(G13,G23,result)
            threehop2[itera,thetai] = sum((result[1]==truth).float())/n

            result = PGM(G1,G2,seeds)
            pgm[itera,thetai] = sum(result==truth)/n

            result = SGM2(G1,G2,seeds)
            sgm[itera,thetai] = sum((result==truth).float())/n


            result = MGCN(G1,G2,seeds)
            mgcn[thetai] = sum((result==truth).float())/n

    seedgnnstd, seedgnn = torch.std_mean(seedgnn,dim=0,unbiased=False)
    onehop6std,onehop6 = torch.std_mean(onehop6,dim=0,unbiased=False)
    twohop3std,twohop3 = torch.std_mean(twohop3,dim=0,unbiased=False)
    threehop2std,threehop2 = torch.std_mean(threehop2,dim=0,unbiased=False)
    pgmstd,pgm = torch.std_mean(pgm,dim=0,unbiased=False)
    sgmstd,sgm = torch.std_mean(sgm,dim=0,unbiased=False)


    theta = [round(i, 4) for i in to_list_if_needed(Theta)]
    seedgnn = [round(i, 4) for i in to_list_if_needed(seedgnn)]
    onehop6 = [round(i, 4) for i in to_list_if_needed(onehop6)]
    twohop3 = [round(i, 4) for i in to_list_if_needed(twohop3)]
    threehop2 = [round(i, 4) for i in to_list_if_needed(threehop2)]
    pgm = [round(i, 4) for i in to_list_if_needed(pgm)]
    sgm = [round(i, 4) for i in to_list_if_needed(sgm)]
    mgcn = [round(i, 4) for i in to_list_if_needed(mgcn)]

    torch.set_printoptions(precision=4)
    print('Accuracy')
    print('theta ='.ljust(10), theta)
    print('SeedGNN = '.ljust(10),seedgnn)
    print('onehop6 ='.ljust(10), onehop6)
    print('twohop3 ='.ljust(10), twohop3)
    print('threehop2 ='.ljust(10), threehop2)
    print('pgm = '.ljust(10), pgm)
    print('sgm = '.ljust(10), sgm)
    print('mgcn = '.ljust(10), mgcn)



if __name__ ==  '__main__':

    parser = argparse.ArgumentParser()
    parser.add_argument('--hid', type=int, default=4)
    parser.add_argument('--num_layers', type=int, default=6)

    args, unknown = parser.parse_known_args()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model = SeedGNN(num_layers=args.num_layers, hid=args.hid).to(device)

    path = "./model/SeedGNN-model.pth"
    model.load_state_dict(torch.load(path))
    for param in model.parameters():
        param.requires_grad = False

    graphA = wiki_adj['G_EN_Adj']
    graphB = wiki_adj['G_FR_Adj']
    wiki_labels = scipy.io.loadmat('/content/drive/MyDrive/wiki_labels.mat')
    wiki_labels = wiki_labels['wiki_labels'][0]
    indices = wiki_labels.argsort()

    graphA = wiki_adj['G_EN_Adj'][indices][:, indices]
    graphB = wiki_adj['G_FR_Adj'][indices][:, indices]

    comm = [370,270,191,212,220, 119]


    L = 6

    number_of_seeds = [0, 5, 50, 150, 250, 382, 500]

    cum_comm = [0]
    print(cum_comm)
    for size in comm:
        cum_comm.append(cum_comm[-1]+size)


    for k in range(len(comm)):
        c_start = cum_comm[k]
        c_end = cum_comm[k+1]
        size = comm[k]
        print(f"Community {k+1} size: " + str(size))
        Theta = [(x / 1382) for x in number_of_seeds]
        graphA_new = graphA[c_start:c_end, c_start:c_end]
        graphB_new = graphB[c_start:c_end, c_start:c_end]
        run(graphA_new, graphB_new, L, Theta, Itera)
        print('-----------------------------------------------')




[0]
Community 1 size: 370
Accuracy
theta =    [0.0, 0.0036, 0.0362, 0.1085, 0.1809, 0.2764, 0.3618]
SeedGNN =  [0.0159, 0.0143, 0.0854, 0.2805, 0.397, 0.5262, 0.6038]
onehop6 =  [0.0078, 0.0084, 0.047, 0.1359, 0.2495, 0.3262, 0.3781]
twohop3 =  [0.0111, 0.01, 0.0559, 0.1265, 0.1738, 0.2019, 0.2211]
threehop2 = [0.0054, 0.0084, 0.0416, 0.0978, 0.1308, 0.147, 0.1462]
pgm =      [0.0027, 0.0046, 0.0132, 0.0446, 0.087, 0.1232, 0.1514]
sgm =      [0.0289, 0.0408, 0.1122, 0.3289, 0.4768, 0.5795, 0.6703]
mgcn =     [0.0027, 0.0054, 0.0378, 0.1108, 0.1811, 0.2757, 0.3595]
-----------------------------------------------
Community 2 size: 270
Accuracy
theta =    [0.0, 0.0036, 0.0362, 0.1085, 0.1809, 0.2764, 0.3618]
SeedGNN =  [0.013, 0.0089, 0.0622, 0.1659, 0.2659, 0.3767, 0.4833]
onehop6 =  [0.0189, 0.0122, 0.0478, 0.1337, 0.2074, 0.293, 0.343]
twohop3 =  [0.0078, 0.0085, 0.0363, 0.0507, 0.0826, 0.0941, 0.123]
threehop2 = [0.0041, 0.0063, 0.0348, 0.0633, 0.0767, 0.0774, 0.0974]
pgm =      [0.00

In [ ]:
wiki_labels = scipy.io.loadmat('/content/drive/MyDrive/wiki_labels.mat')['wiki_labels'][0]
#wiki_labels = wiki_labels['wiki_labels'][0]
indices = wiki_labels.argsort()

In [ ]:
wiki_labels

array([2, 2, 4, ..., 1, 1, 4])

In [ ]:
import sys
import os.path as osp
import numpy as np
import time

import argparse
import torch

from SeedGNN import SeedGNN
from MGCN import MGCN
from GMAlgorithms import MultiHop, PGM, SGM2

def SynGraph_from_numpy(A, B, theta, degreesort=False):
    """
    Parameters:
        A, B: numpy.ndarray, adjacency matrices of two graphs (n x n)
        theta: float, proportion of nodes to use as seeds (0 <= theta <= 1)
        degreesort: bool, if True, uses degree sort for permutation

    Returns:
        G1: torch.FloatTensor, permuted adjacency matrix for graph 1
        G2: torch.FloatTensor, adjacency matrix for graph 2 (possibly permuted if degreesort=True)
        seeds: list of two 1D torch.LongTensor, [indices in G1, matching indices in G2]
        truth: 1D torch.LongTensor, ground-truth mapping from rows of G1 to rows of G2
    """

    A_tensor = torch.from_numpy(A).float()
    B_tensor = torch.from_numpy(B).float()
    n = A_tensor.size(0)
    assert B_tensor.size(0) == n and B_tensor.size(1) == n, "A and B must be n x n matrices"

    if degreesort:

        degree = A_tensor.sum(0)
        _, indices = torch.sort(degree, descending=True)
        G1 = A_tensor[indices][:, indices]
        G2 = B_tensor[indices][:, indices]
        truth = n - 1 - torch.arange(n)
    else:
        truth = torch.randperm(n)
        G1 = A_tensor[truth][:, truth]
        G2 = B_tensor

    numseeds = int(n * theta)
    indices = torch.randperm(n)[:numseeds]
    seeds = [indices, truth[indices]]

    return G1, G2, seeds, truth

def to_list_if_needed(value):

    if isinstance(value, list):
        return value

    if hasattr(value, 'tolist'):
        return value.tolist()

    return value



def generate_y(num_nodes, truth):
    oneton = torch.arange(num_nodes)
    return [oneton, truth]

def test(test_dataset):
    model.eval()

    total_correct = 0
    total_node = 0
    for data in test_dataset:


        G1 = data[0]
        G2 = data[1]
        seeds = data[2]
        truth = data[3]
        num_nodes = G1.shape[0]

        Y_L, _ = model(G1,G2,seeds)

        y = generate_y(num_nodes, truth)
        correct = model.acc(Y_L,y)
        total_correct += correct
        total_node += num_nodes
    return total_correct/total_node

def new_generate_array_comm(l, x):
    total = sum(l)
    ratio = x / total
    l2 = [min(int(size * ratio), size) for size in l]
    for i in range(len(l)):
        if l2[i] == l[i]:
            if l2[i] > 0:
                l2[i] -= 1
            else:
                l2[i] += 1
    current_sum = sum(l2)
    difference = x - current_sum

    fractional_parts = [(size * ratio) - int(size * ratio) for size in l]
    indices_sorted = sorted(
        range(len(l)),
        key=lambda i: (fractional_parts[i], -l[i]),
        reverse=True
    )

    for i in indices_sorted:
        if difference <= 0:
            break
        if abs(l2[i] + 1 - l[i]) >= 1 and l2[i] < l[i]:
            l2[i] += 1
            difference -= 1

    return l2


def run(graphA, graphB, L,Theta,Itera):
    n = len(graphA)
    seedgnn = torch.zeros(Itera,len(Theta))
    onehop6 = torch.zeros(Itera,len(Theta))
    twohop3 = torch.zeros(Itera,len(Theta))
    threehop2 = torch.zeros(Itera,len(Theta))
    pgm = torch.zeros(Itera,len(Theta))
    sgm = torch.zeros(Itera,len(Theta))
    mgcn = torch.zeros(len(Theta))

    for thetai, theta in enumerate(Theta):
        for itera in range(Itera):
            datasets = []
            G1, G2, seeds, truth = SynGraph_from_numpy(graphA,graphB,theta)
            datasets = [(G1, G2, seeds, truth)]
            eyes = torch.eye(n)
            G1 = G1
            G12 = (( ((torch.mm(G1, G1))>0).float() - G1 - eyes)>0).float()
            G22 = (( ((torch.mm(G2, G2))>0).float() - G2 - eyes)>0).float()
            G13 = (( ((torch.mm(G12, G1))>0).float() - G12 - G1 - eyes)>0).float()
            G23 = (( ((torch.mm(G22, G2))>0).float() - G22 - G2 - eyes)>0).float()

            SeedGNN
            seedgnn[itera,thetai] = test(datasets)


            result = seeds
            for _ in range(L):
                result = MultiHop(G1,G2,result)
            onehop6[itera,thetai] = sum((result[1]==truth).float())/n

            result = seeds
            for _ in range(int(L/2)):
                result = MultiHop(G12,G22,result)
            twohop3[itera,thetai] = sum((result[1]==truth).float())/n

            result = seeds
            for _ in range(int(L/3)):
                result = MultiHop(G13,G23,result)
            threehop2[itera,thetai] = sum((result[1]==truth).float())/n

            result = PGM(G1,G2,seeds)
            pgm[itera,thetai] = sum(result==truth)/n

            result = SGM2(G1,G2,seeds)
            sgm[itera,thetai] = sum((result==truth).float())/n


            result = MGCN(G1,G2,seeds)
            mgcn[thetai] = sum((result==truth).float())/n

    seedgnnstd, seedgnn = torch.std_mean(seedgnn,dim=0,unbiased=False)
    onehop6std,onehop6 = torch.std_mean(onehop6,dim=0,unbiased=False)
    twohop3std,twohop3 = torch.std_mean(twohop3,dim=0,unbiased=False)
    threehop2std,threehop2 = torch.std_mean(threehop2,dim=0,unbiased=False)
    pgmstd,pgm = torch.std_mean(pgm,dim=0,unbiased=False)
    sgmstd,sgm = torch.std_mean(sgm,dim=0,unbiased=False)


    theta = [round(i, 4) for i in to_list_if_needed(Theta)]
    seedgnn = [round(i, 4) for i in to_list_if_needed(seedgnn)]
    onehop6 = [round(i, 4) for i in to_list_if_needed(onehop6)]
    twohop3 = [round(i, 4) for i in to_list_if_needed(twohop3)]
    threehop2 = [round(i, 4) for i in to_list_if_needed(threehop2)]
    pgm = [round(i, 4) for i in to_list_if_needed(pgm)]
    sgm = [round(i, 4) for i in to_list_if_needed(sgm)]
    mgcn = [round(i, 4) for i in to_list_if_needed(mgcn)]

    torch.set_printoptions(precision=4)
    print('Accuracy')
    print('theta ='.ljust(10), theta)
    print('SeedGNN = '.ljust(10),seedgnn)
    print('onehop6 ='.ljust(10), onehop6)
    print('twohop3 ='.ljust(10), twohop3)
    print('threehop2 ='.ljust(10), threehop2)
    print('pgm = '.ljust(10), pgm)
    print('sgm = '.ljust(10), sgm)
    print('mgcn = '.ljust(10), mgcn)


if __name__ ==  '__main__':

    parser = argparse.ArgumentParser()
    parser.add_argument('--hid', type=int, default=4)
    parser.add_argument('--num_layers', type=int, default=6)

    args, unknown = parser.parse_known_args()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model = SeedGNN(num_layers=args.num_layers, hid=args.hid).to(device)

    path = "./model/SeedGNN-model.pth"
    model.load_state_dict(torch.load(path))
    for param in model.parameters():
        param.requires_grad = False

    start = time.time()
    L = 6
    number_of_seeds = [1, 5, 10, 20, 50, 75, 100, 150, 200]
    Theta = [x / 300 for x in number_of_seeds]

    Itera = 10

    graphA = matrices3[(0.3, 0, 'A')]
    graphB = matrices3[(0.3, 0, 'B')]

    number_of_seeds = [1, 5, 10, 20, 50, 75, 100, 150, 200]
    comm = [100,100,100]
    cum_comm = [0]

    for size in comm:
        cum_comm.append(cum_comm[-1]+size)


    for k in range(len(comm)):
        c_start = cum_comm[k]
        c_end = cum_comm[k+1]
        size = comm[k]
        print(f"Community {k+1} size: " + str(graphA[c_start:c_end, c_start:c_end].shape))
        Theta = [x / 300 for x in number_of_seeds]
        graphA_new = graphA[c_start:c_end, c_start:c_end]
        graphB_new = graphB[c_start:c_end, c_start:c_end]
        run(graphA_new, graphB_new, L, Theta, Itera)
        print('-----------------------------------------------')

Community 1 size: (100, 100)
Accuracy
theta =    [0.0033, 0.0167, 0.0333, 0.0667, 0.1667, 0.25, 0.3333, 0.5, 0.6667]
SeedGNN =  [0.007, 0.021, 0.048, 0.084, 0.237, 0.386, 0.51, 0.738, 0.863]
onehop6 =  [0.025, 0.024, 0.051, 0.084, 0.186, 0.286, 0.332, 0.541, 0.766]
twohop3 =  [0.012, 0.016, 0.028, 0.044, 0.133, 0.211, 0.289, 0.571, 0.795]
threehop2 = [0.013, 0.004, 0.006, 0.013, 0.007, 0.01, 0.013, 0.011, 0.013]
pgm =      [0.01, 0.015, 0.01, 0.024, 0.04, 0.043, 0.041, 0.089, 0.094]
sgm =      [0.019, 0.043, 0.072, 0.126, 0.328, 0.576, 0.73, 0.976, 0.997]
mgcn =     [0.01, 0.03, 0.04, 0.06, 0.17, 0.25, 0.33, 0.5, 0.66]
-----------------------------------------------
Community 2 size: (100, 100)
Accuracy
theta =    [0.0033, 0.0167, 0.0333, 0.0667, 0.1667, 0.25, 0.3333, 0.5, 0.6667]
SeedGNN =  [0.007, 0.018, 0.041, 0.084, 0.2, 0.316, 0.419, 0.64, 0.834]
onehop6 =  [0.014, 0.034, 0.06, 0.081, 0.178, 0.279, 0.315, 0.446, 0.553]
twohop3 =  [0.01, 0.017, 0.021, 0.046, 0.085, 0.199, 0.239, 0.

In [ ]:
import numpy as np

# correlation 0.3
n_seed = [1, 5, 10, 20, 50, 75, 100, 150, 200]
CESSNA = [0.0383, 0.074, 0.1497, 0.302, 0.9973, 1.0, 1.0, 1.0, 1.0]
SeedGNN =  [0.012, 0.034, 0.0653, 0.1353, 0.466, 0.7133, 0.7913, 0.816, 0.8497]
onehop6 =  [0.0187, 0.042, 0.0713, 0.12, 0.4207, 0.9993, 1.0, 1.0, 1.0]
twohop3 =  [0.0097, 0.025, 0.0467, 0.0997, 0.3403, 0.743, 0.987, 1.0, 1.0]
threehop2 = [0.0023, 0.0043, 0.005, 0.0057, 0.012, 0.009, 0.014, 0.0203, 0.0263]
pgm =      [0.004, 0.0077, 0.0127, 0.0167, 0.0223, 0.034, 0.0363, 0.0893, 0.2017]
sgm =      [0.036, 0.0723, 0.1253, 0.2613, 1.0, 1.0, 1.0, 1.0, 1.0]
mgcn =     [0.0033, 0.02, 0.04, 0.07, 0.17, 0.25, 0.3333, 0.5, 0.67]


# correlation 0.6
n_seed = [1, 5, 10, 20, 50, 75, 100, 150, 200]
CESSNA = [0.9957, 1.0, 1.0, 1.0, 0.9967, 1.0, 1.0, 1.0, 1.0]
SeedGNN =  [0.0133, 0.4563, 1.0, 1.0, 0.9907, 0.9407, 0.9137, 0.9483, 0.976]
onehop6 =  [0.0243, 0.053, 0.8587, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
twohop3 =  [0.015, 0.0497, 0.2977, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
threehop2 = [0.0027, 0.0053, 0.0043, 0.0117, 0.017, 0.027, 0.0263, 0.0333, 0.0473]
pgm =      [0.0057, 0.0087, 0.0157, 0.0367, 0.2, 0.747, 0.9123, 0.981, 0.9813]
sgm =      [0.0717, 0.9697, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
mgcn =     [0.0067, 0.02, 0.0333, 0.07, 0.17, 0.2533, 0.3367, 0.5067, 0.67]

In [ ]:
CESSNA_0_6 = [np.float64(0.9956521739130434),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.9960000000000001),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0)]